In [0]:
%run ../config

### 型を変換した新テーブルを作成

In [0]:
# silverテーブルを作成
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {silver_audit_table_path}
    (
        `event_id` STRING,
        `event_time` TIMESTAMP,
        `action_name` STRING,
        `resource_name` STRING,
        `source_ip` STRING,
        `user` STRING,
        `user_email` STRING,
        `user_name` STRING,
        `request_params` STRING,
        _datasource STRING,
        _ingest_timestamp TIMESTAMP
    )
    """
)

# テーブル作り直し
spark.sql(
    f"""
    REPLACE TABLE {silver_audit_table_path} (
        `event_id` STRING,
        `event_time` TIMESTAMP,
        `action_name` STRING,
        `resource_name` STRING,
        `source_ip` STRING,
        `user` STRING,
        `user_email` STRING,
        `user_name` STRING,
        `request_params` STRING,
        _datasource STRING,
        _ingest_timestamp TIMESTAMP
    )
    """
)

## ブロンズテーブルを抽出し、型変換

In [0]:
df = spark.sql(
    f"""
    SELECT
        event_id,
        CAST(event_time AS TIMESTAMP) AS event_time,
        action_name,
        resource_name,
        source_ip,
        user,
        request_params,
        _datasource,
        _ingest_timestamp
    FROM {bronze_audit_table_path}
    """
)

In [0]:
# 処理後の結果を確認
df.display()

## クレンジング


### クレンジング前の確認
- `user`, `request_params` がJSONとして読めない行
- 必須列null行


In [0]:
from pyspark.sql import functions as F

quality_df = df.withColumn(
    "is_required_null",
    F.col("event_id").isNull()
    | F.col("event_time").isNull()
    | F.col("action_name").isNull(),
)

# 先頭に空白があるかどうかを判定するカラムを追加
for col_name in ["action_name", "resource_name", "request_params", "user", "source_ip"]:
    quality_df = quality_df.withColumn(
        f"{col_name}_starts_with_space", F.col(col_name).startswith(" ")
    )

summary = quality_df.select(
    F.count("*").alias("row_count"),
    F.sum(F.col("is_required_null").cast("int")).alias("required_null_rows"),
    *[
        F.sum(F.col(f"{col}_starts_with_space").cast("int")).alias(
            f"{col}_starts_with_space_count"
        )
        for col in [
            "action_name",
            "resource_name",
            "request_params",
            "user",
            "source_ip",
        ]
    ],
)

display(summary)

### 先頭空白除去(trim処理):

- `action_name`, `resource_name`, `request_params`, `source_ip` へ適用
値の両端にある空白文字を削除しましょう。


In [0]:
from pyspark.sql import functions as F

### null削除

- audit: `event_time`, `action_name`, `user_email` がnullの行を除外

### user JSON を展開（email/nameを列にする）

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

### 重複削除

- `event_id` を一意にして重複排除

In [0]:
# カラム順序を整列
df = df.select(
    "event_id",
    "event_time",
    "action_name",
    "resource_name",
    "source_ip",
    "user",
    "user_email",
    "user_name",
    "request_params",
    "_datasource",
    "_ingest_timestamp",
)

シルバーテーブルに書き込む

In [0]:
(df.write.format("delta").mode("overwrite").saveAsTable(silver_audit_table_path))

# spark.sql(f"optimize {silver_audit_table_path} zorder by (Timestamp)")
display(spark.sql(f"select * from {silver_audit_table_path}"))

TODO：リキッドクラスタリング